# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### Nguyên bản 100% mã nguồn từ: https://github.com/SYNEXAN/AI-Urban-Flood-Traffic
Triển khai Gradio trên Kaggle GPU (Tesla T4) + Ngrok Domain cố định.
Giữ nguyên 100% luồng chạy real-time streaming, SCSG-Net, ByteTrack và backend của source gốc.

In [ ]:
# 1. Cài đặt các thư viện cần thiết cho Gradio và YOLO
!pip install -q "huggingface_hub<0.26.0" "gradio==4.44.0" "starlette<0.40.0" "fastapi<0.116.0" "pydantic<2.10.0" pyngrok ultralytics opencv-python-headless pillow pandas lapx

In [ ]:
# 2. Clone trực tiếp 100% mã nguồn gốc từ GitHub SYNEXAN/AI-Urban-Flood-Traffic
import os
repo_url = "https://github.com/SYNEXAN/AI-Urban-Flood-Traffic.git"
print(f"⬇️ Đang clone nguyên bản từ {repo_url}...")
!rm -rf AI-Urban-Flood-Traffic
!git clone {repo_url}
%cd AI-Urban-Flood-Traffic
!ls -la

In [ ]:
# 3. Tự động tải Trọng số mô hình Deep Learning (Flood Seg & Vehicle Detection)
import os, urllib.request, time

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

for path, url, min_size in models_info:
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        continue
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path}.")
                break
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(2)

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/

In [ ]:
# 4. Khởi động Ngrok Tunnel & Chạy Gradio App Nguyên Bản (100% Backend Gốc)
import subprocess
from pyngrok import ngrok

NGROK_AUTHTOKEN = "3LNVtpalQtpB352VIkzga6oB3dg_5qLYQiojiE9bG79gf5Liq"
NGROK_DOMAIN = "hush-harmless-erased.ngrok-free.dev"

if NGROK_AUTHTOKEN:
    try:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
    except Exception as e:
        print(f"Warning set_auth_token: {e}")

try:
    ngrok.kill()
except Exception:
    pass

public_url = None
try:
    print("Connecting via internal endpoint (https://default.internal)...")
    tunnel = ngrok.connect(7860, url="https://default.internal", pooling_enabled=True)
    public_url = f"https://{NGROK_DOMAIN}"
    print(f"Connected to internal endpoint successfully! Public domain: {public_url}")
except Exception as e1:
    print(f"Strategy 1 error: {e1}")
    try:
        print(f"Connecting via static url https://{NGROK_DOMAIN} with pooling...")
        tunnel = ngrok.connect(7860, url=f"https://{NGROK_DOMAIN}", pooling_enabled=True)
        public_url = tunnel.public_url
    except Exception as e2:
        print(f"Strategy 2 error: {e2}")
        tunnel = ngrok.connect(7860)
        public_url = tunnel.public_url

print("=" * 70)
print("🚀 GRADIO WEB APP (SYNEXAN/AI-Urban-Flood-Traffic) NGUYÊN BẢN ĐÃ SẴN SÀNG!")
print(f"👉 NGROK PUBLIC URL: {public_url}")
print("=" * 70)

# Khởi chạy app.py nguyên bản 100% từ GitHub source (luồng real-time stream)
subprocess.run(["python", "app.py"])